# XAI Healthcare DSS: Model Training & Performance Proof
**Project Context:** Seminar Demonstration  
**Models:** XGBoost & Random Forest  
**Goal:** Prove the genuineness of medical predictions and XAI explanations.

---

## 1. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import shap
import os

# Set paths relative to project root
TRAIN_PATH = '../data/Training.csv'
METADATA_PATH = './metadata.json'

## 2. Data Ingestion
We load the `Training.csv` which contains 132 unique medical symptoms mapped to various diseases.

In [ ]:
df = pd.read_csv(TRAIN_PATH).loc[:, ~pd.read_csv(TRAIN_PATH).columns.str.contains('^Unnamed')]
print(f"Dataset Shape: {df.shape}")
display(df.head())

## 3. Training the Diagnostic Engine
We use **XGBoost** for its high accuracy in tabular data and its compatibility with SHAP for explainability.

In [ ]:
X = df.drop('prognosis', axis=1)
y = df['prognosis']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print("Training XGBoost Model...")
model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

## 4. Proving Genuineness: Confusion Matrix
A confusion matrix shows exactly where the model succeeded and where it failed. This proves the results are not hardcoded.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 8))
sns.heatmap(cm[:10, :10], annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_[:10], yticklabels=le.classes_[:10])
plt.title("Confusion Matrix (Top 10 Classes) - Statistical Proof")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 5. XAI: SHAP Explanations
SHAP (SHapley Additive exPlanations) is used to show which symptoms drove the prediction. This is the 'X' in XAI.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Calculate mean absolute SHAP for global importance
if isinstance(shap_values, list):
    mean_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
else:
    mean_shap = np.abs(shap_values).mean(axis=0)

feature_importance = pd.DataFrame({'feature': X.columns, 'importance': mean_shap})
top_features = feature_importance.sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=top_features, palette='viridis')
plt.title("Top 10 Feature Importance (SHAP values)")
plt.show()

## 6. Closing the Loop: Sync with Website
The metrics you see above are exported to `metadata.json`, which the web dashboard then reads dynamically.

In [ ]:
import json
with open(METADATA_PATH, 'r') as f:
    meta = json.load(f)

print(f"Last Sync with Dashboard: {meta['last_updated']}")
print(f"Accuracy in Package: {meta['metrics']['accuracy'] * 100:.2f}%")